In [ ]:
#@title Diagnóstico — GPU visível e build do PyTorch
import subprocess
# nvidia-smi existe se há GPU alocada no runtime
print("=== nvidia-smi ===")
print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout[:600] or "SEM GPU")

import torch
print("\n=== PyTorch ===")
print(f"versão: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
print(f"build CUDA: {torch.version.cuda}")   # None = build CPU-only

In [ ]:
#@title Setup + reprodutibilidade (determinismo PyTorch)
# Fixa TODAS as fontes de não-determinismo antes de qualquer modelo.
# Sem isto, dois runs do mesmo LSTM divergem e a comparação nível 4 vs 5
# (núcleo da tese) fica irreproduzível. A multi-semente (decisão 7) varia
# a semente DE PROPÓSITO; esta função garante que, FIXADA uma semente,
# o resultado é bit-a-bit reproduzível.

from pathlib import Path
PROJECT = Path(r'D:\tcc_ibov_corpo_completo')
DIRS = {k: PROJECT / v for k, v in {
    'raw': 'data/raw', 'processed': 'data/processed',
    'models': 'models', 'reports': 'reports', 'utils': 'utils',
}.items()}
for d in DIRS.values(): d.mkdir(parents=True, exist_ok=True)

import sys, os, random
sys.path.insert(0, str(DIRS['utils']))
import importlib, tcc_utils; importlib.reload(tcc_utils)
from tcc_utils import make_target, walk_forward_splits

import numpy as np
import pandas as pd
import torch

def set_seed(seed: int):
    """Trava todas as fontes de aleatoriedade p/ reprodutibilidade bit-a-bit.
    Chamada no início de CADA treino, com a semente daquele run."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Determinismo CUDA (custa um pouco de performance, mas garante replicação)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
print(f"PyTorch: {torch.__version__}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# Teste do determinismo: duas inicializações com a mesma semente devem ser idênticas
set_seed(42); a = torch.randn(3, device=DEVICE)
set_seed(42); b = torch.randn(3, device=DEVICE)
print(f"\nDeterminismo OK: {torch.allclose(a, b)}  (deve ser True)")

In [ ]:
#@title Embargo na walk_forward_splits -- ja incluso no tcc_utils canonico
# A funcao walk_forward_splits (tcc_utils, gravada pelo 00_setup) ja aceita
# embargo>0, usado aqui para o LSTM (embargo=21=lookback) sem afetar os
# baselines (embargo=0, default). Esta celula so valida o comportamento.

import importlib, tcc_utils; importlib.reload(tcc_utils)
from tcc_utils import walk_forward_splits, make_target

precos = pd.read_parquet(DIRS['processed'] / 'bvsp_clean.parquet')
estudo = precos.loc['2019-01-01':'2024-12-31'].copy()
estudo['y'] = make_target(estudo); estudo = estudo.dropna(subset=['y'])

splits_lstm = walk_forward_splits(estudo.index, n_splits=5, min_train=252,
                                  expanding=True, embargo=21)
print(f"\n{len(splits_lstm)} folds com embargo=21:")
for k, (tr, te) in enumerate(splits_lstm, 1):
    gap = te[0] - tr[-1] - 1
    print(f"  fold {k}: treino termina idx {tr[-1]} ({estudo.index[tr[-1]].date()}) | "
          f"teste comeca idx {te[0]} ({estudo.index[te[0]].date()}) | gap={gap}")
    assert gap == 21, f"embargo incorreto no fold {k}: gap={gap}"
print("\nEmbargo OK: gap de exatamente 21 dias entre treino e teste em todos os folds.")


In [ ]:
#@title Preparação de sequências (janelas lookback=21) + scaling POR FOLD
# Converte a tabela (dias × features) em sequências (amostra × 21 × features).
# CRÍTICO: o StandardScaler é ajustado SÓ no treino de cada fold e aplicado ao
# teste. Ajustar na série inteira vazaria a distribuição futura. Por isso o
# scaling NÃO acontece aqui — acontece dentro do loop de treino, por fold.
# Esta célula só monta as janelas cruas e o alvo alinhado.

from sklearn.preprocessing import StandardScaler

LOOKBACK = 21

feat = pd.read_parquet(DIRS['processed'] / 'features_preco.parquet')
FEAT_COLS = [c for c in feat.columns if c != 'y']
X_raw = feat[FEAT_COLS].values.astype(np.float32)    # (n_dias, 18)
y_raw = feat['y'].values.astype(np.float32)          # (n_dias,) retorno D+1

def make_sequences(X, y, lookback):
    """Janela deslizante: sequência[i] = X[i-lookback+1 : i+1] (21 dias até i,
    inclusive), alvo[i] = y[i] (retorno de D+1 já embutido em y). Retorna
    índices originais p/ alinhar com os folds do walk-forward."""
    seqs, targets, idx = [], [], []
    for i in range(lookback - 1, len(X)):
        seqs.append(X[i - lookback + 1 : i + 1])   # (21, 18)
        targets.append(y[i])
        idx.append(i)                               # posição do dia-alvo
    return np.array(seqs), np.array(targets), np.array(idx)

X_seq, y_seq, seq_idx = make_sequences(X_raw, y_raw, LOOKBACK)
print(f"Sequências: {X_seq.shape}  (esperado ~{len(feat)-LOOKBACK+1} x 21 x 18)")
print(f"Alvos: {y_seq.shape} | índices originais: {seq_idx.min()}..{seq_idx.max()}")

# Recalcula folds COM embargo sobre o índice ORIGINAL de dias (não de sequências).
# A conversão fold-de-dias -> fold-de-sequências acontece no loop de treino,
# filtrando seq_idx pelos índices de treino/teste de cada fold.
splits_lstm = walk_forward_splits(feat.index, n_splits=5, min_train=252,
                                  expanding=True, embargo=LOOKBACK)

# Verificação: quantas sequências caem em cada fold (treino/teste)?
print(f"\nSequências por fold (após embargo):")
for k, (tr, te) in enumerate(splits_lstm, 1):
    tr_set, te_set = set(tr), set(te)
    n_tr = sum(1 for i in seq_idx if i in tr_set)
    n_te = sum(1 for i in seq_idx if i in te_set)
    print(f"  fold {k}: treino {n_tr} seqs | teste {n_te} seqs")

# Persistir p/ o loop de treino não recalcular
np.savez(DIRS['processed'] / 'lstm_sequences.npz',
         X_seq=X_seq, y_seq=y_seq, seq_idx=seq_idx)
print("\nSalvo: lstm_sequences.npz")

In [ ]:
#@title Modelo LSTM (1 camada, hidden=16, dropout=0.4) → regressor escalar
import torch.nn as nn

class LSTMRegressor(nn.Module):
    """LSTM parcimonioso p/ dataset pequeno (~230-1050 seqs/fold).
    - 1 camada, hidden=16: razão params/amostra controlada.
    - dropout na saída do LSTM (não entre camadas, pois só há 1).
    - cabeça linear -> 1 escalar: retorno log de D+1 (regressão, loss MSE).
    Entrada: (batch, 21, n_features). Saída: (batch,)."""
    def __init__(self, n_features, hidden=16, dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(input_size=n_features, hidden_size=hidden,
                            num_layers=1, batch_first=True)
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        # out: (batch, seq, hidden); pegamos o ÚLTIMO passo temporal (h do dia t)
        out, (h_n, c_n) = self.lstm(x)
        last = out[:, -1, :]          # (batch, hidden) — estado no fim da janela
        last = self.dropout(last)
        return self.head(last).squeeze(-1)   # (batch,)

# Teste de sanidade: forward com batch fake, confere shapes e determinismo
set_seed(42)
n_feat = 18
model_test = LSTMRegressor(n_feat).to(DEVICE)
n_params = sum(p.numel() for p in model_test.parameters())
print(f"Parâmetros: {n_params}")
print(f"Razão params/amostra (fold 1, 232 seqs): {n_params/232:.1f}")

xb = torch.randn(8, 21, n_feat, device=DEVICE)
out = model_test(xb)
print(f"Saída: {out.shape}  (esperado torch.Size([8]))")

# Determinismo: mesma semente -> mesma inicialização -> mesma saída
set_seed(42); m1 = LSTMRegressor(n_feat).to(DEVICE); o1 = m1(xb)
set_seed(42); m2 = LSTMRegressor(n_feat).to(DEVICE); o2 = m2(xb)
print(f"Determinismo do modelo: {torch.allclose(o1, o2)}  (deve ser True)")

In [ ]:
#@title Nível 4 — LSTM (só preço) · treino multi-semente com scaling POR FOLD
# Junta todas as decisões: lookback=21, embargo=21, StandardScaler ajustado
# SÓ no treino de cada fold, arquitetura parcimoniosa, N=5 sementes, 75 épocas
# fixas (sem early stopping -> sem consumir dados p/ validação). Avaliação nas
# 3 dimensões (MAE/RMSE, DA/F1/MCC, Brier via calibração posterior).
# ANTI-VAZAMENTO: scaler.fit só em X de treino; nada de teste toca o ajuste.

import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (mean_absolute_error, f1_score,
                             matthews_corrcoef, brier_score_loss)
import time

data = np.load(DIRS['processed'] / 'lstm_sequences.npz')
X_seq, y_seq, seq_idx = data['X_seq'], data['y_seq'], data['seq_idx']

HIDDEN, DROPOUT, EPOCHS, LR, WD, BATCH = 16, 0.4, 75, 1e-3, 1e-4, 32
SEEDS = [0, 1, 2, 3, 4]

def rmse(a, b): return np.sqrt(np.mean((a - b) ** 2))
def da(a, b):   return (np.sign(a) == np.sign(b)).mean()

def scale_sequences(X_tr, X_te):
    """StandardScaler ajustado no TREINO (achatando tempo×features), aplicado
    a ambos. Ajuste sobre (n_tr*21, 18) -> um scaler global de features."""
    n_f = X_tr.shape[-1]
    sc = StandardScaler().fit(X_tr.reshape(-1, n_f))
    X_tr_s = sc.transform(X_tr.reshape(-1, n_f)).reshape(X_tr.shape)
    X_te_s = sc.transform(X_te.reshape(-1, n_f)).reshape(X_te.shape)
    return X_tr_s.astype(np.float32), X_te_s.astype(np.float32)

def treina_avalia(X_tr, y_tr, X_te, seed):
    set_seed(seed)
    model = LSTMRegressor(X_tr.shape[-1], HIDDEN, DROPOUT).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WD)
    lossf = nn.MSELoss()

    Xtr = torch.tensor(X_tr, device=DEVICE); ytr = torch.tensor(y_tr, device=DEVICE)
    n = len(Xtr)
    model.train()
    for ep in range(EPOCHS):
        perm = torch.randperm(n, device=DEVICE)   # shuffle (determinístico via seed)
        for i in range(0, n, BATCH):
            b = perm[i:i+BATCH]
            opt.zero_grad()
            loss = lossf(model(Xtr[b]), ytr[b])
            loss.backward()
            opt.step()

    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X_te, device=DEVICE)).cpu().numpy()
    return pred

registros = []
preds_media = {}   # fold -> média das previsões contínuas sobre sementes

for k, (tr, te) in enumerate(splits_lstm, 1):
    t0 = time.time()
    tr_set, te_set = set(tr), set(te)
    mask_tr = np.array([i in tr_set for i in seq_idx])
    mask_te = np.array([i in te_set for i in seq_idx])

    X_tr, y_tr = X_seq[mask_tr], y_seq[mask_tr]
    X_te, y_te = X_seq[mask_te], y_seq[mask_te]
    X_tr_s, X_te_s = scale_sequences(X_tr, X_te)   # scaling POR FOLD

    # hold-out temporal interno p/ calibração (últimos 20% do treino)
    n_cal = int(len(X_tr_s) * 0.2)
    preds_seeds = []
    for s in SEEDS:
        pred = treina_avalia(X_tr_s, y_tr, X_te_s, s)
        preds_seeds.append(pred)

        # Calibração: mapeia retorno previsto -> P(alta) via isotônica no treino.
        # Treina calibrador nas previsões do PRÓPRIO treino (hold-out interno).
        pred_tr = treina_avalia(X_tr_s[:-n_cal], y_tr[:-n_cal],
                                X_tr_s[-n_cal:], s)
        iso = IsotonicRegression(out_of_bounds='clip').fit(
            pred_tr, (y_tr[-n_cal:] > 0).astype(int))
        p_prob = iso.predict(pred)

        y_bin = (y_te > 0).astype(int); pred_dir = (p_prob > 0.5).astype(int)
        registros.append({
            'fold': k, 'seed': s,
            'MAE': mean_absolute_error(y_te, pred), 'RMSE': rmse(y_te, pred),
            'DA': da(y_te, pred),
            'F1': f1_score(y_bin, pred_dir, zero_division=0),
            'MCC': matthews_corrcoef(y_bin, pred_dir),
            'Brier': brier_score_loss(y_bin, p_prob),
        })
    preds_media[k] = np.mean(preds_seeds, axis=0)
    print(f"fold {k}: {len(X_tr_s)} tr seqs | {time.time()-t0:.0f}s")

reg_df = pd.DataFrame(registros)
metric_cols = ['MAE', 'RMSE', 'DA', 'F1', 'MCC', 'Brier']
print("\n=== Média sobre 5 sementes, por fold ===")
print(reg_df.groupby('fold')[metric_cols].mean().round(4).to_string())
print("\n=== Desvio sobre 5 sementes, por fold ===")
print(reg_df.groupby('fold')[metric_cols].std().round(4).to_string())

resumo = reg_df.groupby('fold')[metric_cols].mean().mean()
desvio = ((reg_df.groupby('fold')[metric_cols].std() ** 2).mean()) ** 0.5
print("\n=== Resumo global (média ± desvio entre sementes) ===")
for m in metric_cols:
    print(f"  {m}: {resumo[m]:+.4f} ± {desvio[m]:.4f}")

np.savez(DIRS['reports'] / 'preds_lstm_preco.npz',
         **{f'lstm_f{k}': v for k, v in preds_media.items()})
reg_df.to_parquet(DIRS['reports'] / 'lstm_preco_full.parquet')
print("\nSalvos: preds_lstm_preco.npz, lstm_preco_full.parquet")

In [ ]:
#@title Nível 5 — LSTM (preço + texto) · PCA(768→8) e scaling POR FOLD
# A TESE. Idêntico ao nível 4 em TUDO (arquitetura, sementes, folds, embargo,
# calibração) exceto: adiciona features textuais ao tensor de entrada.
# Isolamento do efeito do texto = diferença nível5 - nível4.
# ANTI-VAZAMENTO: PCA dos embeddings + scaler ajustados SÓ no treino de cada
# fold. Features de texto do dia t (notícias até corte 17h) -> prevê t+1.

from sklearn.decomposition import PCA

# --- Carrega e ALINHA texto com preço ---
feat_preco = pd.read_parquet(DIRS['processed'] / 'features_preco.parquet')
feat_texto = pd.read_parquet(DIRS['processed'] / 'features_texto_bertimbau_corpo.parquet')

# Alinha texto ao índice do preço (features_preco já é o estudo limpo, 1492 dias)
feat_texto = feat_texto.reindex(feat_preco.index)
assert feat_texto['emb_000'].notna().all(), "texto com NaN após alinhar — investigar"
print(f"Alinhamento OK: preço {feat_preco.shape} | texto {feat_texto.shape}")

EMB_COLS = [f'emb_{i:03d}' for i in range(768)]
SIG_COLS = ['n_noticias', 'emb_dispersao']       # sinais agregados de baixa dim
PRICE_COLS = [c for c in feat_preco.columns if c != 'y']

X_price = feat_preco[PRICE_COLS].values.astype(np.float32)   # (n, 18)
X_emb   = feat_texto[EMB_COLS].values.astype(np.float32)     # (n, 768)
X_sig   = feat_texto[SIG_COLS].values.astype(np.float32)     # (n, 2)
y_raw   = feat_preco['y'].values.astype(np.float32)

K_PCA = 8

def make_seq_multi(mats, y, lookback):
    """Igual à make_sequences, mas concatena múltiplas matrizes de features
    por dia. As matrizes já vêm alinhadas linha a linha (mesmo índice)."""
    X = np.concatenate(mats, axis=1)
    seqs, targets, idx = [], [], []
    for i in range(lookback - 1, len(X)):
        seqs.append(X[i - lookback + 1 : i + 1]); targets.append(y[i]); idx.append(i)
    return np.array(seqs), np.array(targets), np.array(idx)

# NÃO aplicamos PCA aqui (vazaria). Montamos sequências com preço+sig+emb CRUS;
# o PCA dos emb acontece por fold, dentro do loop, ANTES de virar sequência.
# Por isso guardamos as 3 matrizes separadas e sequenciamos dentro do fold.

HIDDEN, DROPOUT, EPOCHS, LR, WD, BATCH = 16, 0.4, 75, 1e-3, 1e-4, 32
SEEDS = [0, 1, 2, 3, 4]
LOOKBACK = 21

def rmse(a, b): return np.sqrt(np.mean((a - b) ** 2))
def da(a, b):   return (np.sign(a) == np.sign(b)).mean()

splits_lstm = walk_forward_splits(feat_preco.index, n_splits=5, min_train=252,
                                  expanding=True, embargo=LOOKBACK)

registros, preds_media, var_explicada = [], {}, []

for k, (tr_days, te_days) in enumerate(splits_lstm, 1):
    t0 = time.time()
    tr_set, te_set = set(tr_days), set(te_days)

    # 1) PCA dos embeddings ajustado SÓ nos dias de TREINO do fold
    tr_day_mask = np.array([i in tr_set for i in range(len(X_emb))])
    pca = PCA(n_components=K_PCA, random_state=0).fit(X_emb[tr_day_mask])
    var_explicada.append(pca.explained_variance_ratio_.sum())
    X_emb_pca = pca.transform(X_emb).astype(np.float32)   # (n, 8) transform em tudo

    # 2) Monta features do dia = [preço(18) | sig(2) | emb_pca(8)] = 28 dims
    X_day = np.concatenate([X_price, X_sig, X_emb_pca], axis=1)

    # 3) Sequências
    seqs, targets, sidx = [], [], []
    for i in range(LOOKBACK - 1, len(X_day)):
        seqs.append(X_day[i-LOOKBACK+1:i+1]); targets.append(y_raw[i]); sidx.append(i)
    seqs, targets, sidx = np.array(seqs), np.array(targets), np.array(sidx)

    mask_tr = np.array([i in tr_set for i in sidx])
    mask_te = np.array([i in te_set for i in sidx])
    X_tr, y_tr = seqs[mask_tr], targets[mask_tr]
    X_te, y_te = seqs[mask_te], targets[mask_te]

    # 4) StandardScaler ajustado no treino (achata tempo×features)
    n_f = X_tr.shape[-1]
    sc = StandardScaler().fit(X_tr.reshape(-1, n_f))
    X_tr = sc.transform(X_tr.reshape(-1, n_f)).reshape(X_tr.shape).astype(np.float32)
    X_te = sc.transform(X_te.reshape(-1, n_f)).reshape(X_te.shape).astype(np.float32)

    n_cal = int(len(X_tr) * 0.2)
    preds_seeds = []
    for s in SEEDS:
        pred = treina_avalia(X_tr, y_tr, X_te, s)   # reusa função do nível 4
        preds_seeds.append(pred)
        pred_tr = treina_avalia(X_tr[:-n_cal], y_tr[:-n_cal], X_tr[-n_cal:], s)
        iso = IsotonicRegression(out_of_bounds='clip').fit(
            pred_tr, (y_tr[-n_cal:] > 0).astype(int))
        p_prob = iso.predict(pred)
        y_bin = (y_te > 0).astype(int); pred_dir = (p_prob > 0.5).astype(int)
        registros.append({
            'fold': k, 'seed': s,
            'MAE': mean_absolute_error(y_te, pred), 'RMSE': rmse(y_te, pred),
            'DA': da(y_te, pred),
            'F1': f1_score(y_bin, pred_dir, zero_division=0),
            'MCC': matthews_corrcoef(y_bin, pred_dir),
            'Brier': brier_score_loss(y_bin, p_prob),
        })
    preds_media[k] = np.mean(preds_seeds, axis=0)
    print(f"fold {k}: {len(X_tr)} tr seqs | PCA var explicada {var_explicada[-1]:.1%} | "
          f"{time.time()-t0:.0f}s")

reg_df5 = pd.DataFrame(registros)
metric_cols = ['MAE', 'RMSE', 'DA', 'F1', 'MCC', 'Brier']
print(f"\nPCA k={K_PCA}: variância explicada média {np.mean(var_explicada):.1%}")
print("\n=== Média sobre 5 sementes, por fold ===")
print(reg_df5.groupby('fold')[metric_cols].mean().round(4).to_string())
print("\n=== Desvio sobre 5 sementes, por fold ===")
print(reg_df5.groupby('fold')[metric_cols].std().round(4).to_string())

resumo = reg_df5.groupby('fold')[metric_cols].mean().mean()
desvio = ((reg_df5.groupby('fold')[metric_cols].std() ** 2).mean()) ** 0.5
print("\n=== Resumo global (média ± desvio) ===")
for m in metric_cols:
    print(f"  {m}: {resumo[m]:+.4f} ± {desvio[m]:.4f}")

np.savez(DIRS['reports'] / 'preds_lstm_texto.npz',
         **{f'lstm_f{k}': v for k, v in preds_media.items()})
reg_df5.to_parquet(DIRS['reports'] / 'lstm_texto_full.parquet')
print("\nSalvos: preds_lstm_texto.npz, lstm_texto_full.parquet")

In [ ]:
#@title Nível 5b — desambiguação: PCA k=30 (retém mais variância)
# CONTROLE: o nulo do k=8 pode ser (a) texto sem sinal ou (b) sinal perdido na
# compressão (k=8 reteve só 45%). k=30 deve reter ~65-75%. Se TAMBÉM der nulo,
# reforça (a). Se melhorar, era compressão. Mesmo código do nível 5, só muda k.
# Reusa: X_price, X_emb, X_sig, y_raw, splits_lstm, treina_avalia (já em memória).

def roda_lstm_texto(k_pca, tag):
    """Roda o LSTM preço+texto com PCA(k_pca). Retorna DataFrame de métricas.
    Idêntico ao nível 5 exceto o k. PCA e scaler ajustados por fold."""
    registros, preds_media, var_exp = [], {}, []
    for kf, (tr_days, te_days) in enumerate(splits_lstm, 1):
        t0 = time.time()
        tr_set, te_set = set(tr_days), set(te_days)
        tr_day_mask = np.array([i in tr_set for i in range(len(X_emb))])

        pca = PCA(n_components=k_pca, random_state=0).fit(X_emb[tr_day_mask])
        var_exp.append(pca.explained_variance_ratio_.sum())
        X_emb_pca = pca.transform(X_emb).astype(np.float32)
        X_day = np.concatenate([X_price, X_sig, X_emb_pca], axis=1)

        seqs, targets, sidx = [], [], []
        for i in range(LOOKBACK - 1, len(X_day)):
            seqs.append(X_day[i-LOOKBACK+1:i+1]); targets.append(y_raw[i]); sidx.append(i)
        seqs, targets, sidx = np.array(seqs), np.array(targets), np.array(sidx)

        mask_tr = np.array([i in tr_set for i in sidx])
        mask_te = np.array([i in te_set for i in sidx])
        X_tr, y_tr = seqs[mask_tr], targets[mask_tr]
        X_te, y_te = seqs[mask_te], targets[mask_te]

        n_f = X_tr.shape[-1]
        sc = StandardScaler().fit(X_tr.reshape(-1, n_f))
        X_tr = sc.transform(X_tr.reshape(-1, n_f)).reshape(X_tr.shape).astype(np.float32)
        X_te = sc.transform(X_te.reshape(-1, n_f)).reshape(X_te.shape).astype(np.float32)

        n_cal = int(len(X_tr) * 0.2)
        preds_seeds = []
        for s in SEEDS:
            pred = treina_avalia(X_tr, y_tr, X_te, s)
            preds_seeds.append(pred)
            pred_tr = treina_avalia(X_tr[:-n_cal], y_tr[:-n_cal], X_tr[-n_cal:], s)
            iso = IsotonicRegression(out_of_bounds='clip').fit(
                pred_tr, (y_tr[-n_cal:] > 0).astype(int))
            p_prob = iso.predict(pred)
            y_bin = (y_te > 0).astype(int); pred_dir = (p_prob > 0.5).astype(int)
            registros.append({
                'fold': kf, 'seed': s,
                'MAE': mean_absolute_error(y_te, pred), 'RMSE': rmse(y_te, pred),
                'DA': da(y_te, pred),
                'F1': f1_score(y_bin, pred_dir, zero_division=0),
                'MCC': matthews_corrcoef(y_bin, pred_dir),
                'Brier': brier_score_loss(y_bin, p_prob),
            })
        preds_media[kf] = np.mean(preds_seeds, axis=0)
        print(f"  fold {kf}: var {var_exp[-1]:.1%} | {time.time()-t0:.0f}s")

    df = pd.DataFrame(registros)
    np.savez(DIRS['reports'] / f'preds_lstm_texto_{tag}.npz',
             **{f'lstm_f{k}': v for k, v in preds_media.items()})
    df.to_parquet(DIRS['reports'] / f'lstm_texto_{tag}_full.parquet')
    return df, np.mean(var_exp)

print("=== PCA k=30 ===")
reg_k30, var30 = roda_lstm_texto(30, 'k30')

metric_cols = ['MAE', 'RMSE', 'DA', 'F1', 'MCC', 'Brier']
resumo30 = reg_k30.groupby('fold')[metric_cols].mean().mean()
desvio30 = ((reg_k30.groupby('fold')[metric_cols].std() ** 2).mean()) ** 0.5

# Comparação direta k=8 vs k=30 vs só-preço (nível 4)
n4 = pd.read_parquet(DIRS['reports'] / 'lstm_preco_full.parquet')
n5_k8 = pd.read_parquet(DIRS['reports'] / 'lstm_texto_full.parquet')
res_n4  = n4.groupby('fold')[metric_cols].mean().mean()
res_k8  = n5_k8.groupby('fold')[metric_cols].mean().mean()

print(f"\nVariância explicada: k=8 ~45% | k=30 ~{var30:.0%}")
print("\n=== Comparação (MCC/DA/RMSE/Brier) ===")
for m in ['MCC', 'DA', 'RMSE', 'Brier']:
    print(f"  {m}: só-preço={res_n4[m]:+.4f} | texto-k8={res_k8[m]:+.4f} | "
          f"texto-k30={resumo30[m]:+.4f} ±{desvio30[m]:.4f}")

print("\n=== k=30 por fold ===")
print(reg_k30.groupby('fold')[metric_cols].mean().round(4).to_string())
print("\nSalvos: preds_lstm_texto_k30.npz, lstm_texto_k30_full.parquet")

In [ ]:
#@title Via 2 — LSTM (preço + sentimento FinBERT) · 4 features densas, SEM PCA
# Sentimento são 4 features densas interpretáveis -> entram DIRETO, sem PCA.
# Tensor: (21, 18+4)=(21,22). Idêntico ao nível 4 em tudo (arquitetura, sementes,
# folds, embargo, scaling, calibração) exceto a adição das 4 features de sentimento.
# Isolamento do efeito = diferença vs nível 4 (só-preço).
# ANTI-VAZAMENTO: sentimento(t) de notícias até corte 17h; scaler por fold.

feat_preco = pd.read_parquet(DIRS['processed'] / 'features_preco.parquet')
feat_sent  = pd.read_parquet(DIRS['processed'] / 'features_sentimento_finbert_corpo.parquet')
feat_sent  = feat_sent.reindex(feat_preco.index)
assert feat_sent.notna().all().all(), "sentimento com NaN após alinhar"

PRICE_COLS = [c for c in feat_preco.columns if c != 'y']
SENT_COLS  = ['sent_polaridade', 'sent_dispersao', 'sent_pos_frac', 'sent_neg_frac']

X_price = feat_preco[PRICE_COLS].values.astype(np.float32)   # (n,18)
X_sent  = feat_sent[SENT_COLS].values.astype(np.float32)     # (n,4)
y_raw   = feat_preco['y'].values.astype(np.float32)
X_day   = np.concatenate([X_price, X_sent], axis=1)          # (n,22)

HIDDEN, DROPOUT, EPOCHS, LR, WD, BATCH = 16, 0.4, 75, 1e-3, 1e-4, 32
SEEDS = [0, 1, 2, 3, 4]; LOOKBACK = 21

def rmse(a, b): return np.sqrt(np.mean((a - b) ** 2))
def da(a, b):   return (np.sign(a) == np.sign(b)).mean()

splits_lstm = walk_forward_splits(feat_preco.index, n_splits=5, min_train=252,
                                  expanding=True, embargo=LOOKBACK)

registros, preds_media = [], {}
for kf, (tr_days, te_days) in enumerate(splits_lstm, 1):
    t0 = time.time()
    tr_set, te_set = set(tr_days), set(te_days)

    seqs, targets, sidx = [], [], []
    for i in range(LOOKBACK - 1, len(X_day)):
        seqs.append(X_day[i-LOOKBACK+1:i+1]); targets.append(y_raw[i]); sidx.append(i)
    seqs, targets, sidx = np.array(seqs), np.array(targets), np.array(sidx)

    mask_tr = np.array([i in tr_set for i in sidx])
    mask_te = np.array([i in te_set for i in sidx])
    X_tr, y_tr = seqs[mask_tr], targets[mask_tr]
    X_te, y_te = seqs[mask_te], targets[mask_te]

    n_f = X_tr.shape[-1]
    sc = StandardScaler().fit(X_tr.reshape(-1, n_f))
    X_tr = sc.transform(X_tr.reshape(-1, n_f)).reshape(X_tr.shape).astype(np.float32)
    X_te = sc.transform(X_te.reshape(-1, n_f)).reshape(X_te.shape).astype(np.float32)

    n_cal = int(len(X_tr) * 0.2)
    preds_seeds = []
    for s in SEEDS:
        pred = treina_avalia(X_tr, y_tr, X_te, s)
        preds_seeds.append(pred)
        pred_tr = treina_avalia(X_tr[:-n_cal], y_tr[:-n_cal], X_tr[-n_cal:], s)
        iso = IsotonicRegression(out_of_bounds='clip').fit(
            pred_tr, (y_tr[-n_cal:] > 0).astype(int))
        p_prob = iso.predict(pred)
        y_bin = (y_te > 0).astype(int); pred_dir = (p_prob > 0.5).astype(int)
        registros.append({
            'fold': kf, 'seed': s,
            'MAE': mean_absolute_error(y_te, pred), 'RMSE': rmse(y_te, pred),
            'DA': da(y_te, pred),
            'F1': f1_score(y_bin, pred_dir, zero_division=0),
            'MCC': matthews_corrcoef(y_bin, pred_dir),
            'Brier': brier_score_loss(y_bin, p_prob),
        })
    preds_media[kf] = np.mean(preds_seeds, axis=0)
    print(f"fold {kf}: {len(X_tr)} tr seqs | {time.time()-t0:.0f}s")

reg_via2 = pd.DataFrame(registros)
metric_cols = ['MAE', 'RMSE', 'DA', 'F1', 'MCC', 'Brier']

# Comparação com nível 4 (só-preço)
n4 = pd.read_parquet(DIRS['reports'] / 'lstm_preco_full.parquet')
res_n4   = n4.groupby('fold')[metric_cols].mean().mean()
res_via2 = reg_via2.groupby('fold')[metric_cols].mean().mean()
dev_via2 = ((reg_via2.groupby('fold')[metric_cols].std() ** 2).mean()) ** 0.5

print("\n=== Via 2 (preço+sentimento) vs nível 4 (só-preço) ===")
for m in ['MCC', 'DA', 'RMSE', 'Brier']:
    print(f"  {m}: só-preço={res_n4[m]:+.4f} | +sentimento={res_via2[m]:+.4f} "
          f"±{dev_via2[m]:.4f} | Δ={res_via2[m]-res_n4[m]:+.4f}")

print("\n=== Via 2 por fold ===")
print(reg_via2.groupby('fold')[metric_cols].mean().round(4).to_string())

np.savez(DIRS['reports'] / 'preds_lstm_via2_sent.npz',
         **{f'lstm_f{k}': v for k, v in preds_media.items()})
reg_via2.to_parquet(DIRS['reports'] / 'lstm_via2_sent_full.parquet')
print("\nSalvos: preds_lstm_via2_sent.npz, lstm_via2_sent_full.parquet")

In [ ]:
#@title Via 1 -- LSTM (preço + embeddings FinBERT) · PCA(768→8) e scaling POR FOLD
# Espelha o nível 5 (BERTimbau) em TUDO -- só troca a fonte dos embeddings.
# Fecha a simetria BERTimbau vs FinBERT prometida na proposta: mesmo k=8
# inicial, mesma arquitetura, sementes, folds, embargo, calibração.
# ANTI-VAZAMENTO: PCA + scaler ajustados SÓ no treino de cada fold.

feat_preco = pd.read_parquet(DIRS['processed'] / 'features_preco.parquet')
feat_texto_fb = pd.read_parquet(DIRS['processed'] / 'features_texto_finbert_corpo.parquet')
feat_texto_fb = feat_texto_fb.reindex(feat_preco.index)
assert feat_texto_fb['emb_000'].notna().all(), "texto FinBERT com NaN após alinhar"
print(f"Alinhamento OK: preço {feat_preco.shape} | texto FinBERT {feat_texto_fb.shape}")

EMB_COLS = [f'emb_{i:03d}' for i in range(768)]
SIG_COLS = ['n_noticias', 'emb_dispersao']
PRICE_COLS = [c for c in feat_preco.columns if c != 'y']

X_price_fb = feat_preco[PRICE_COLS].values.astype(np.float32)
X_emb_fb   = feat_texto_fb[EMB_COLS].values.astype(np.float32)
X_sig_fb   = feat_texto_fb[SIG_COLS].values.astype(np.float32)
y_raw_fb   = feat_preco['y'].values.astype(np.float32)

HIDDEN, DROPOUT, EPOCHS, LR, WD, BATCH = 16, 0.4, 75, 1e-3, 1e-4, 32
SEEDS = [0, 1, 2, 3, 4]; LOOKBACK = 21

def rmse(a, b): return np.sqrt(np.mean((a - b) ** 2))
def da(a, b):   return (np.sign(a) == np.sign(b)).mean()

splits_lstm = walk_forward_splits(feat_preco.index, n_splits=5, min_train=252,
                                  expanding=True, embargo=LOOKBACK)

def roda_lstm_texto_fonte(X_price, X_sig, X_emb, y_raw, k_pca, tag):
    """Igual à roda_lstm_texto (nível 5b), mas parametrizada pela FONTE dos
    embeddings (BERTimbau ou FinBERT) -- reusa treina_avalia já em memória."""
    registros, preds_media, var_exp = [], {}, []
    for kf, (tr_days, te_days) in enumerate(splits_lstm, 1):
        t0 = time.time()
        tr_set, te_set = set(tr_days), set(te_days)
        tr_day_mask = np.array([i in tr_set for i in range(len(X_emb))])

        pca = PCA(n_components=k_pca, random_state=0).fit(X_emb[tr_day_mask])
        var_exp.append(pca.explained_variance_ratio_.sum())
        X_emb_pca = pca.transform(X_emb).astype(np.float32)
        X_day = np.concatenate([X_price, X_sig, X_emb_pca], axis=1)

        seqs, targets, sidx = [], [], []
        for i in range(LOOKBACK - 1, len(X_day)):
            seqs.append(X_day[i-LOOKBACK+1:i+1]); targets.append(y_raw[i]); sidx.append(i)
        seqs, targets, sidx = np.array(seqs), np.array(targets), np.array(sidx)

        mask_tr = np.array([i in tr_set for i in sidx])
        mask_te = np.array([i in te_set for i in sidx])
        X_tr, y_tr = seqs[mask_tr], targets[mask_tr]
        X_te, y_te = seqs[mask_te], targets[mask_te]

        n_f = X_tr.shape[-1]
        sc = StandardScaler().fit(X_tr.reshape(-1, n_f))
        X_tr = sc.transform(X_tr.reshape(-1, n_f)).reshape(X_tr.shape).astype(np.float32)
        X_te = sc.transform(X_te.reshape(-1, n_f)).reshape(X_te.shape).astype(np.float32)

        n_cal = int(len(X_tr) * 0.2)
        preds_seeds = []
        for s in SEEDS:
            pred = treina_avalia(X_tr, y_tr, X_te, s)
            preds_seeds.append(pred)
            pred_tr = treina_avalia(X_tr[:-n_cal], y_tr[:-n_cal], X_tr[-n_cal:], s)
            iso = IsotonicRegression(out_of_bounds='clip').fit(
                pred_tr, (y_tr[-n_cal:] > 0).astype(int))
            p_prob = iso.predict(pred)
            y_bin = (y_te > 0).astype(int); pred_dir = (p_prob > 0.5).astype(int)
            registros.append({
                'fold': kf, 'seed': s,
                'MAE': mean_absolute_error(y_te, pred), 'RMSE': rmse(y_te, pred),
                'DA': da(y_te, pred),
                'F1': f1_score(y_bin, pred_dir, zero_division=0),
                'MCC': matthews_corrcoef(y_bin, pred_dir),
                'Brier': brier_score_loss(y_bin, p_prob),
            })
        preds_media[kf] = np.mean(preds_seeds, axis=0)
        print(f"  fold {kf}: var {var_exp[-1]:.1%} | {time.time()-t0:.0f}s")

    df = pd.DataFrame(registros)
    np.savez(DIRS['reports'] / f'preds_lstm_{tag}.npz',
             **{f'lstm_f{k}': v for k, v in preds_media.items()})
    df.to_parquet(DIRS['reports'] / f'lstm_{tag}_full.parquet')
    return df, np.mean(var_exp)

metric_cols = ['MAE', 'RMSE', 'DA', 'F1', 'MCC', 'Brier']

print("=== Via 1 -- FinBERT emb, PCA k=8 ===")
reg_fb_k8, var_fb_k8 = roda_lstm_texto_fonte(X_price_fb, X_sig_fb, X_emb_fb, y_raw_fb, 8, 'via1_finbert_k8')

resumo_fb8 = reg_fb_k8.groupby('fold')[metric_cols].mean().mean()
desvio_fb8 = ((reg_fb_k8.groupby('fold')[metric_cols].std() ** 2).mean()) ** 0.5

n4 = pd.read_parquet(DIRS['reports'] / 'lstm_preco_full.parquet')
res_n4 = n4.groupby('fold')[metric_cols].mean().mean()

print(f"\nVariância explicada (FinBERT k=8): {var_fb_k8:.1%}")
print("\n=== Via 1 (FinBERT k=8) vs nível 4 (só-preço) ===")
for m in ['MCC', 'DA', 'RMSE', 'Brier']:
    print(f"  {m}: só-preço={res_n4[m]:+.4f} | +emb FinBERT k8={resumo_fb8[m]:+.4f} "
          f"±{desvio_fb8[m]:.4f} | Δ={resumo_fb8[m]-res_n4[m]:+.4f}")

print("\n=== Via 1 (k=8) por fold ===")
print(reg_fb_k8.groupby('fold')[metric_cols].mean().round(4).to_string())
print("\nSalvos: preds_lstm_via1_finbert_k8.npz, lstm_via1_finbert_k8_full.parquet")


In [ ]:
#@title Via 1b -- desambiguação PCA k=30 (FinBERT) + comparação final texto
# Mesmo controle do nível 5b, aplicado à fonte FinBERT: k=8 reteve pouca
# variância -> testamos k=30 para descartar perda de sinal por compressão.
# Fecha com a tabela comparativa das 4 variantes de texto (BERTimbau/FinBERT
# × k=8/k=30) contra o baseline só-preço.

print("=== Via 1b -- FinBERT emb, PCA k=30 ===")
reg_fb_k30, var_fb_k30 = roda_lstm_texto_fonte(
    X_price_fb, X_sig_fb, X_emb_fb, y_raw_fb, 30, 'via1_finbert_k30')

resumo_fb30 = reg_fb_k30.groupby('fold')[metric_cols].mean().mean()
desvio_fb30 = ((reg_fb_k30.groupby('fold')[metric_cols].std() ** 2).mean()) ** 0.5

print(f"\nVariância explicada (FinBERT k=30): {var_fb_k30:.1%}")
print("\n=== Via 1b (FinBERT k=30) vs nível 4 (só-preço) ===")
for m in ['MCC', 'DA', 'RMSE', 'Brier']:
    print(f"  {m}: só-preço={res_n4[m]:+.4f} | +emb FinBERT k30={resumo_fb30[m]:+.4f} "
          f"±{desvio_fb30[m]:.4f} | Δ={resumo_fb30[m]-res_n4[m]:+.4f}")

# --- Tabela comparativa final: as 4 variantes de texto (embeddings) ---
n5_k8  = pd.read_parquet(DIRS['reports'] / 'lstm_texto_full.parquet')
n5_k30 = pd.read_parquet(DIRS['reports'] / 'lstm_texto_k30_full.parquet')

def resumo_desvio(df):
    r = df.groupby('fold')[metric_cols].mean().mean()
    d = ((df.groupby('fold')[metric_cols].std() ** 2).mean()) ** 0.5
    return r, d

r_bert8, d_bert8   = resumo_desvio(n5_k8)
r_bert30, d_bert30 = resumo_desvio(n5_k30)

print("\n=== Comparação final -- MCC ± desvio entre sementes ===")
linhas = [
    ("só-preço (nível 4)",        res_n4['MCC'], None),
    ("BERTimbau k=8 (nível 5)",   r_bert8['MCC'], d_bert8['MCC']),
    ("BERTimbau k=30 (nível 5b)", r_bert30['MCC'], d_bert30['MCC']),
    ("FinBERT emb k=8 (via 1)",   resumo_fb8['MCC'], desvio_fb8['MCC']),
    ("FinBERT emb k=30 (via 1b)", resumo_fb30['MCC'], desvio_fb30['MCC']),
]
for nome, mcc, dev in linhas:
    dev_str = f" ± {dev:.4f}" if dev is not None else ""
    print(f"  {nome:<28s}: MCC {mcc:+.4f}{dev_str}")


In [ ]:
#@title Via 3 -- LSTM (preço + Copom) · 3 features densas, SEM PCA
# Comunicados do Copom (decisão de política monetária), coletados via API de
# dados abertos do BCB (scripts/coleta_copom.py, 49 reuniões 2018-2024) e
# classificados por zero-shot (MoritzLaurer/mDeBERTa-v3-base-mnli-xnli) em
# viés hawkish (aperto) vs dovish (afrouxamento). Sanidade: correlação entre
# o score zero-shot e a direção REAL da decisão de Selic = +0,615 -- o
# classificador capta o viés real, não ruído.
# ANTI-VAZAMENTO: comunicado liberado ~18h30 no dia da reunião (após o
# fechamento) -> cai no PRÓXIMO pregão via a mesma regra de corte (17h) usada
# para notícias (assign_trading_day). O regime de política monetária é
# tratado como persistente entre reuniões (forward-fill) -- convenção padrão
# em finanças macro: o mercado precifica o último viés conhecido do BC até
# a próxima decisão. Nível absoluto da Selic é EXCLUÍDO das features (mesma
# lógica do volume: carrega tendência estrutural, não sinal de curto prazo)
# -- entram só copom_hawkish_score, direcao_ultima e dias_desde_copom.

feat_preco = pd.read_parquet(DIRS['processed'] / 'features_preco.parquet')
feat_copom  = pd.read_parquet(DIRS['processed'] / 'features_copom.parquet')
feat_copom  = feat_copom.reindex(feat_preco.index)
assert feat_copom.notna().all().all(), "Copom com NaN após alinhar"

PRICE_COLS = [c for c in feat_preco.columns if c != 'y']
COPOM_COLS = ['copom_hawkish_score', 'direcao_ultima', 'dias_desde_copom']

X_price = feat_preco[PRICE_COLS].values.astype(np.float32)   # (n,18)
X_copom = feat_copom[COPOM_COLS].values.astype(np.float32)   # (n,3)
y_raw   = feat_preco['y'].values.astype(np.float32)
X_day   = np.concatenate([X_price, X_copom], axis=1)          # (n,21)

HIDDEN, DROPOUT, EPOCHS, LR, WD, BATCH = 16, 0.4, 75, 1e-3, 1e-4, 32
SEEDS = [0, 1, 2, 3, 4]; LOOKBACK = 21

def rmse(a, b): return np.sqrt(np.mean((a - b) ** 2))
def da(a, b):   return (np.sign(a) == np.sign(b)).mean()

splits_lstm = walk_forward_splits(feat_preco.index, n_splits=5, min_train=252,
                                  expanding=True, embargo=LOOKBACK)

registros, preds_media = [], {}
for kf, (tr_days, te_days) in enumerate(splits_lstm, 1):
    t0 = time.time()
    tr_set, te_set = set(tr_days), set(te_days)

    seqs, targets, sidx = [], [], []
    for i in range(LOOKBACK - 1, len(X_day)):
        seqs.append(X_day[i-LOOKBACK+1:i+1]); targets.append(y_raw[i]); sidx.append(i)
    seqs, targets, sidx = np.array(seqs), np.array(targets), np.array(sidx)

    mask_tr = np.array([i in tr_set for i in sidx])
    mask_te = np.array([i in te_set for i in sidx])
    X_tr, y_tr = seqs[mask_tr], targets[mask_tr]
    X_te, y_te = seqs[mask_te], targets[mask_te]

    n_f = X_tr.shape[-1]
    sc = StandardScaler().fit(X_tr.reshape(-1, n_f))
    X_tr = sc.transform(X_tr.reshape(-1, n_f)).reshape(X_tr.shape).astype(np.float32)
    X_te = sc.transform(X_te.reshape(-1, n_f)).reshape(X_te.shape).astype(np.float32)

    n_cal = int(len(X_tr) * 0.2)
    preds_seeds = []
    for s in SEEDS:
        pred = treina_avalia(X_tr, y_tr, X_te, s)
        preds_seeds.append(pred)
        pred_tr = treina_avalia(X_tr[:-n_cal], y_tr[:-n_cal], X_tr[-n_cal:], s)
        iso = IsotonicRegression(out_of_bounds='clip').fit(
            pred_tr, (y_tr[-n_cal:] > 0).astype(int))
        p_prob = iso.predict(pred)
        y_bin = (y_te > 0).astype(int); pred_dir = (p_prob > 0.5).astype(int)
        registros.append({
            'fold': kf, 'seed': s,
            'MAE': mean_absolute_error(y_te, pred), 'RMSE': rmse(y_te, pred),
            'DA': da(y_te, pred),
            'F1': f1_score(y_bin, pred_dir, zero_division=0),
            'MCC': matthews_corrcoef(y_bin, pred_dir),
            'Brier': brier_score_loss(y_bin, p_prob),
        })
    preds_media[kf] = np.mean(preds_seeds, axis=0)
    print(f"fold {kf}: {len(X_tr)} tr seqs | {time.time()-t0:.0f}s")

reg_via3 = pd.DataFrame(registros)
metric_cols = ['MAE', 'RMSE', 'DA', 'F1', 'MCC', 'Brier']

n4 = pd.read_parquet(DIRS['reports'] / 'lstm_preco_full.parquet')
res_n4   = n4.groupby('fold')[metric_cols].mean().mean()
res_via3 = reg_via3.groupby('fold')[metric_cols].mean().mean()
dev_via3 = ((reg_via3.groupby('fold')[metric_cols].std() ** 2).mean()) ** 0.5

print("\n=== Via 3 (preço+Copom) vs nível 4 (só-preço) ===")
for m in ['MCC', 'DA', 'RMSE', 'Brier']:
    print(f"  {m}: só-preço={res_n4[m]:+.4f} | +Copom={res_via3[m]:+.4f} "
          f"±{dev_via3[m]:.4f} | Δ={res_via3[m]-res_n4[m]:+.4f}")

print("\n=== Via 3 por fold ===")
print(reg_via3.groupby('fold')[metric_cols].mean().round(4).to_string())

np.savez(DIRS['reports'] / 'preds_lstm_via3_copom.npz',
         **{f'lstm_f{k}': v for k, v in preds_media.items()})
reg_via3.to_parquet(DIRS['reports'] / 'lstm_via3_copom_full.parquet')
print("\nSalvos: preds_lstm_via3_copom.npz, lstm_via3_copom_full.parquet")
